In [1]:
%load_ext autoreload
%autoreload 2


In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [6]:
documents_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
len(documents_llm)

113

In [7]:
documents_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [8]:
documents = documents_llm

In [9]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [10]:
from pydantic import BaseModel
class Questions(BaseModel):
    questions:list[str]
    

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
data_gen_instructions

"You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."

In [11]:
from dotenv import load_dotenv
from google import genai
load_dotenv()
client = genai.Client()

In [12]:
import json

user_prompt = json.dumps(doc)

In [13]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=user_prompt,  # just the user message as a string
    config={
        "system_instruction": data_gen_instructions,  # this replaces the "developer" role
        "response_mime_type": "application/json",
        "response_schema": Questions,
    },
)


In [14]:
result= response.parsed


In [15]:
print(result.questions)

['Is it too late for me to register for the course now?', 'Can I still get a certificate if I sign up for the course this late?', 'What is the policy on joining late if I still want to submit a final project?', 'Is there any deadline I need to hit for the certificate if I join right now?', 'Do I have a chance at finishing the program for credit if I start today?']


In [11]:
from evaluation_utils import llm_structured

In [17]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late for me to start the course if I just found it?', "Can I catch up and still participate even though I'm joining late?", 'What do I need to do to get a certificate if I enroll now?', 'Are there any deadlines I should know about for my project if I sign up today?', 'Can I still earn a completion certificate if I join the program at this point?']


In [18]:
usage.prompt_token_count, usage.candidates_token_count

(174, 106)

In [19]:
from evaluation_utils import calc_price

In [20]:
cost = calc_price(usage)

cost

{'input_cost': 0.000261, 'output_cost': 0.000954, 'total_cost': 0.001215}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late for me to start the course if I just found it?',
  'document': '74eb249bbf'},
 {'question': "Can I catch up and still participate even though I'm joining late?",
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get a certificate if I enroll now?',
  'document': '74eb249bbf'},
 {'question': 'Are there any deadlines I should know about for my project if I sign up today?',
  'document': '74eb249bbf'},
 {'question': 'Can I still earn a completion certificate if I join the program at this point?',
  'document': '74eb249bbf'}]

In [22]:
from evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
sad = "sad"

In [12]:
import pandas as pd

df = pd.read_csv("data/ground_truth-new.csv")
df_lessons = pd.read_csv("data/ground-truth-data.csv")
ground_truth = df.to_dict(orient="records")
ground_lies= df.to_dict(orient="records")

In [13]:
# Shows only cells that changed, stacking 'self' and 'other' values
ground_truth[0:5]


[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'}]

In [14]:
ground_lies[0:5]


[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'}]

In [15]:
from ingest import load_faq_data,build_index


In [16]:
documents = load_faq_data()
documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
index = build_index(documents_llm)

In [17]:
index 

In [18]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [19]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [20]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [21]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '610ccb23c0',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code s

In [22]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [23]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [24]:
def compute_relevance_text(q):
    relevance = []
    document_id = q["document"]
    results = text_search(q["question"])
    
    for result in results:
        relevance.append(int(result["id"] == document_id))
    return relevance

In [25]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [26]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [27]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [28]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [29]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [30]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [31]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance




In [32]:


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total



In [33]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [34]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [36]:
def hit_rate(relevance):
    cnt = 0
    for row in relevance: 
        if 1 in row :
            cnt+=1
    return cnt/len(relevance)

In [37]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [38]:
print(hit_rate(example))

0.9333333333333333


In [39]:
#MRR MEAN RECIPROCAL RATE  like the hit rate but laso consider the postion of the right answer 
# the lower you go on the list the lower the score goes (MXN) complixity

In [40]:
total_score  = 0.0 
for row in example:
    for rank in range(len(row)):
        if row[rank]==1:
            total_score = total_score + 1 / (rank+1)
            break ## to find only the first occarunce 
total_score

12.333333333333332

In [41]:
total_score/len(example)

0.8222222222222222

In [42]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank]==1:
                total_score = total_score + 1/(1+rank)
    return total_score / (len(relevance))
    

In [43]:
mrr(relevance_total)

0.5736286919831224

In [44]:
mrr(example)

0.8222222222222222

In [45]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth,search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [46]:
evaluate(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5736286919831224}

In [47]:
evaluate(ground_lies,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5736286919831224}

In [63]:
def search_boost(query, question_boost):
    boost_dict = {"question":question_boost,"section":0.5}
    return index.search(
        query,
        num_results= 5,
        boost_dict=boost_dict
    )

In [64]:
query

NameError: name 'query' is not defined

In [65]:

for boost in [0.5 , 1.0 , 3.0 , 5.0 , 10.0]:
    result = evaluate(ground_truth,
                     lambda query , boost=boost :search_boost(query,boost))


  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

In [69]:
def search_boosts(query,question_boost, answer_boost,section_boost):
    boost_dict = {"question":question_boost,
                  "answer":answer_boost,
                  "section":section_boost
                 }
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

    

In [70]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [71]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr",ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.754430,0.652110
18,2.0,4.0,0.1,0.754430,0.651688
3,1.0,2.0,0.1,0.749367,0.649916
35,5.0,10.0,0.5,0.749367,0.649916
19,2.0,4.0,0.2,0.749367,0.649916
33,5.0,10.0,0.1,0.749367,0.649620
20,2.0,4.0,0.5,0.746835,0.648101
4,1.0,2.0,0.2,0.746835,0.647679
7,1.0,4.0,0.2,0.754430,0.636287
6,1.0,4.0,0.1,0.754430,0.635781


In [72]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )